In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.dates as mdates
import pickle
import seaborn as sns
sns.set(style="whitegrid")


import platform
system_name = platform.system()
if system_name == 'Windows':
    plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows 用黑体
elif system_name == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # Mac 用 Arial Unicode

plt.rcParams['axes.unicode_minus'] = False # 解决负号显示问题

import warnings
from scipy.stats import ConstantInputWarning

warnings.filterwarnings("ignore", category=ConstantInputWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

##########################################################################

def calc_ic(panel, factor_col, ret_col="ret_next"):
    df = panel[["date", factor_col, ret_col]].dropna()
    ic = df.groupby("date").apply(
        lambda x: x[factor_col].corr(x[ret_col], method="spearman")
    )
    ic.name = factor_col
    return ic

def ic_stats(ic):
    ic = ic.dropna()
    ic_mean = ic.mean()
    ic_std = ic.std()

    if pd.isna(ic_std) or ic_std == 0:
        tstat = np.nan
    else:
        tstat = ic_mean / ic_std * np.sqrt(len(ic))

    return pd.Series({
        "IC_mean": ic_mean,
        "IC_std": ic_std,
        "tstat": tstat,
        "N": len(ic)
    })

def csrank(x):
    return x.rank(pct=True) - 0.5



In [2]:
##########################################################################

data = pd.read_csv(r"E:\26Spring\Final\code\data_with_event_bools.csv")

data = data.sort_values(["ticker", "date"]).copy()
data["date"] = pd.to_datetime(data["date"])
data["ret_next"] = data.groupby("ticker")["AdjClose"].shift(-1) / data["AdjClose"] - 1

data["year"] = data["date"].dt.year

liq_year = (
    data.groupby(["year", "ticker"])["amount"]
    .mean()
    .reset_index()
    .rename(columns={"amount": "liq"})
)

liq_year["threshold"] = (
    liq_year.groupby("year")["liq"]
    .transform(lambda x: x.quantile(0.8))
)

liq_top = liq_year[liq_year["liq"] >= liq_year["threshold"]]

data = data.merge(
    liq_top[["year", "ticker"]],
    on=["year", "ticker"],
    how="inner"
)



In [3]:
##########################################################################

ticker_counts = data.groupby('ticker').size().reset_index(name='rows')
ticker_counts_bb = data.groupby('ticker')['buyback_amount'].count().reset_index(name='rows')

merged = pd.merge(ticker_counts, ticker_counts_bb, on='ticker', how='left', suffixes=('_total', '_bb'))
merged["ratio"] = merged["rows_bb"] / merged["rows_total"]
merged.sort_values("ratio", ascending=False)



,ticker,rows_total,rows_bb,ratio
186,2423.hk,185,143,0.772973
4,0005.hk,5237,3604,0.688180
264,9987.hk,489,249,0.509202
32,0300.hk,205,104,0.507317
111,1276.hk,17,7,0.411765
...,...,...,...,...
70,0817.hk,246,0,0.000000
24,0241.hk,1618,0,0.000000
71,0823.hk,1173,0,0.000000
26,0267.hk,984,0,0.000000


In [4]:
##########################################################################

merged['ratio_weight'] = merged['ratio'] / merged['ratio'].sum()
merged = merged.sort_values('ratio_weight', ascending=False)
merged['cum_weight'] = merged['ratio_weight'].cumsum()

top = merged[(merged['cum_weight'] <= 0.8) & (merged['rows_total'] > 100)]

top_tickers = top["ticker"].unique()
filtered_data = data[data["ticker"].isin(top_tickers)].copy()



In [5]:
top

,ticker,rows_total,rows_bb,ratio,ratio_weight,cum_weight
186,2423.hk,185,143,0.772973,0.103640,0.103640
4,0005.hk,5237,3604,0.688180,0.092271,0.195910
264,9987.hk,489,249,0.509202,0.068273,0.264184
32,0300.hk,205,104,0.507317,0.068021,0.332204
240,6690.hk,1158,360,0.310881,0.041683,0.429096
20,0151.hk,492,134,0.272358,0.036518,0.465614
113,1299.hk,2645,683,0.258223,0.034622,0.500236
153,1919.hk,1224,266,0.217320,0.029138,0.529374
91,1024.hk,1133,236,0.208297,0.027928,0.557302
60,0700.hk,2645,534,0.201890,0.027069,0.584372


In [6]:
##########################################################################

nonnull_ratio = filtered_data.groupby('ticker')['buyback_amount'].apply(
    lambda x: (~x.isna()).mean()  
).reset_index(name='nonnull_ratio')

nonnull_ratio.sort_values('nonnull_ratio', ascending=False)



,ticker,nonnull_ratio
17,2423.hk,0.772973
0,0005.hk,0.688180
20,9987.hk,0.509202
2,0300.hk,0.507317
18,6690.hk,0.310881
1,0151.hk,0.272358
10,1299.hk,0.258223
14,1919.hk,0.217320
8,1024.hk,0.208297
5,0700.hk,0.201890


In [7]:
##########################################################################

with pd.HDFStore(r'E:\26Spring\4080\data\hk_shares.h5') as store:
    shares = store.get(store.keys()[0]).reset_index()

shares["ticker"] = shares["order_book_id"].str[1:5] + ".hk"
shares["date"] = pd.to_datetime(shares["date"])

filtered_data["ticker"] = filtered_data["ticker"].str.lower()
filtered_data["date"] = pd.to_datetime(filtered_data["date"])

share_col = "total"  

shares_daily = (
    shares.sort_values("date")
    .groupby(["date", "ticker"], as_index=False)
    .last()
)

filtered_data = filtered_data.merge(
    shares_daily[["date", "ticker", share_col]].rename(columns={share_col: "shares_total"}),
    on=["date", "ticker"],
    how="left"
)

filtered_data["mktcp_new"] = filtered_data["close"] * filtered_data["shares_total"]


In [8]:
filtered_data

,date,ticker,AdjClose,close,open,high,low,volume,amount,turnover,...,hit_corporate_action,hit_cancellation,hit_irrelevant_repurchase,has_any_event,n_event_types,n_repurchase_announcements,ret_next,year,shares_total,mktcp_new
0,2015-01-02,0005.hk,609.385700,74.00,73.60,74.10,73.50,8614012.0,6.353929e+05,NaN,...,False,False,False,False,0,0,-0.014865,2015,1.921787e+10,1.422123e+12
1,2015-01-05,0005.hk,600.327300,72.90,73.15,73.35,72.80,21629040.0,1.579934e+06,NaN,...,False,False,False,False,0,0,-0.015775,2015,1.921787e+10,1.400983e+12
2,2015-01-06,0005.hk,590.857100,71.75,71.80,71.85,71.35,35110417.0,2.514288e+06,NaN,...,False,False,False,False,0,0,-0.010453,2015,1.921787e+10,1.378882e+12
3,2015-01-07,0005.hk,584.680900,71.00,70.50,71.10,70.45,35378291.0,2.505166e+06,NaN,...,False,False,False,False,0,0,0.002817,2015,1.921787e+10,1.364469e+12
4,2015-01-08,0005.hk,586.327900,71.20,71.05,71.40,71.00,21320949.0,1.518474e+06,NaN,...,False,False,False,False,0,0,-0.001405,2015,1.921787e+10,1.368313e+12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30192,2025-09-24,9992.hk,265.758032,258.80,264.00,265.60,255.00,10970912.0,2.836469e+09,1.593308,...,False,False,False,False,0,0,0.011592,2025,1.342943e+09,3.475537e+11
30193,2025-09-25,9992.hk,268.838689,261.80,256.00,264.00,252.80,7969649.0,2.074524e+09,1.157434,...,False,False,False,False,0,0,0.016043,2025,1.342943e+09,3.515825e+11
30194,2025-09-26,9992.hk,273.151609,266.00,261.20,273.40,257.20,13667020.0,3.637677e+09,1.984865,...,False,False,False,False,0,0,-0.015789,2025,1.342943e+09,3.572229e+11
30195,2025-09-29,9992.hk,268.838689,261.80,261.00,264.80,254.00,13745485.0,3.572928e+09,1.996260,...,False,False,False,False,0,0,0.019099,2025,1.342943e+09,3.515825e+11


In [9]:
filtered_data = filtered_data.sort_values(["ticker", "date"]).copy()

filtered_data["buyback_amount_filled"] = filtered_data["buyback_amount"].fillna(0)

filtered_data["buyback_1y"] = (
    filtered_data.groupby("ticker")["buyback_amount_filled"]
    .transform(lambda s: s.rolling(window=252, min_periods=1).sum())
)

filtered_data["buyback_2y"] = (
    filtered_data.groupby("ticker")["buyback_amount_filled"]
    .transform(lambda s: s.rolling(window=504, min_periods=1).sum())
)

filtered_data["buyback_3y"] = (
    filtered_data.groupby("ticker")["buyback_amount_filled"]
    .transform(lambda s: s.rolling(window=756, min_periods=1).sum())
)

filtered_data["bb_yield_1y"] = filtered_data["buyback_1y"] / filtered_data["mktcp_new"]
filtered_data["bb_yield_2y"] = filtered_data["buyback_2y"] / filtered_data["mktcp_new"]
filtered_data["bb_yield_3y"] = filtered_data["buyback_3y"] / filtered_data["mktcp_new"]

filtered_data["bb_share_1y"] = filtered_data["buyback_1y"] / filtered_data["AdjClose"] 
filtered_data["bb_share_2y"] = filtered_data["buyback_2y"] / filtered_data["AdjClose"]
filtered_data["bb_share_3y"] = filtered_data["buyback_3y"] / filtered_data["AdjClose"]


In [10]:
panel = filtered_data.sort_values(["date", "ticker"]).copy()

test_factors = [
    "bb_yield_1y",
    "bb_yield_2y",
    "bb_yield_3y",
    "bb_share_1y",
    "bb_share_2y",
    "bb_share_3y",
    "buyback_1y",
    "buyback_2y",
    "buyback_3y"
]

ic_summary = pd.concat(
    [ic_stats(calc_ic(panel, f, "ret_next")).rename(f) for f in test_factors],
    axis=1
).T

print(ic_summary.sort_values("IC_mean", ascending=False))

              IC_mean    IC_std     tstat       N
buyback_1y   0.010349  0.356438  1.488378  2628.0
buyback_2y   0.009854  0.347516  1.453669  2628.0
bb_share_1y  0.008158  0.360046  1.161553  2628.0
bb_yield_1y  0.007477  0.348574  1.099554  2628.0
buyback_3y   0.006636  0.346115  0.982926  2628.0
bb_yield_2y  0.005143  0.339266  0.777133  2628.0
bb_share_2y  0.005027  0.353018  0.729994  2628.0
bb_share_3y  0.003319  0.353872  0.480794  2628.0
bb_yield_3y  0.003292  0.341820  0.493683  2628.0
